# Table 11 & Table 12: Model Performance Before and After Code Adjustments

## Objective
In this experiment, we evaluate the performance of different models on the **base game** task before and after resolving the leakage issue. This corresponds to **Table 11** and **Table 12** in our paper.

- **Table 11**: Detailed comparison of model performance metrics before and after leakage issue is resolved.
- **Table 12**: A comparison of models prone to leakage before and after code adjustments.  

---

## Methodology

- **Models Evaluated:**  
  - LLaMA-2-13B  
  - Meta-Llama-3-8B  
  - Ministral-8B  
  - Mixtral-8x7B  
  - Phi-3.5-mini  
  - Qwen2.5-7B  

- **Variations Tested:**  
  - **Original Code:** The initial implementation with potential leakage issues.  
  - **Our Code:** The improved implementation with adjustments to prevent leakage.  

- **Evaluation Metrics:**  
  - **% 5/6-way agreement**  
  - **% 6-way agreement**  
  - **% Any agreement**  
  - **% Wrong deals**  
  - **% Leakage**  
  - **% Failed games**  

Each metric helps us assess the overall effectiveness of the models under different conditions.

In [5]:
import eval_utils as evaluation
import os
import json
import numpy as np
import pandas as pd
from IPython.display import display

raw_path = '../our_games_descriptions/base/output'

models = [
    'Llama-2-13b-chat-hf',
    'Meta-Llama-3-8B-Instruct',
    'Ministral-8B-Instruct-2410',
    'Mixtral-8x7B-Instruct-v0.1',
    'Phi-3.5-mini-instruct',
    'Qwen2.5-7B-Instruct'
]

variations = [
    'original_code',
    'our_code'
]

results = {}
ISSUES_NUM = 5
AGENTS_NUM = 6

for variation in variations:
    for model in models:

        # EXTRACT INFORMATION FROM GAME
        directory = os.path.join(raw_path,variation, model)
        agents, role_to_agents, incentive_to_agents = evaluation.load_setup(directory, AGENTS_NUM, num_issues=ISSUES_NUM)
        answers_files = [ os.path.join(directory,filename) for filename in os.listdir(directory) if filename.startswith("history")]

        num_rounds = 0
        for file_ in answers_files:
            answers = json.load(open(file_))
            _num_rounds = len(answers['rounds'])
            num_rounds = max(num_rounds, _num_rounds)


        # Track statistics
        feasible_in_last_step = 0
        accepted_by_all_in_last_step = 0
        contained_feasible_deal = 0
        wrong_deals_percentages = []
        successful_games = 0
        leaked_deals = 0
        total_rounds = 0
        leaked_games = 0

        # Loop through all answer files (each represents a game)
        for file_ in answers_files:
            answers = json.load(open(file_))
            
            if len(answers['rounds']) != num_rounds:
                print(f"WARNING: Game {file_} has a different number of rounds")
                continue
            total_rounds += len(answers['rounds'])
            successful_games += 1

            # Extract deals for this game
            feasible_found = False

            # Extract the name of the first player (p1) to validate feasibility throughout the game
            p1_name = answers['rounds'][0]['agent']

            wrong_deals = 0
            total_deals = 0
            prev_leaked_deals = leaked_deals
            
            for i, round_ in enumerate(answers['rounds']):
                name, answer = round_['agent'], round_['public_answer']
                deal_unformatted, issues_suggested = evaluation.extract_deal(answer, ISSUES_NUM)

                try:
                    deal = evaluation.format_deal(deal_unformatted, ISSUES_NUM)
                except:
                    print(f"Error in game {file_} round {i}")
                    continue

                if issues_suggested >= ISSUES_NUM:
                    wrong_deals += evaluation.is_wrong(agents, deal, agent_name=name)
                    total_deals += 1

                # Check if the deal was feasible at any point (Deal must have been proposed by p1)
                if evaluation.is_feasible(agents, deal) and name == p1_name:
                    feasible_found = True

                # Check for leakage
                leaked_deals += 1 if evaluation.contains_leak(answer) else 0

            wrong_deals_percentage = (wrong_deals / total_deals) * 100
            wrong_deals_percentages.append(wrong_deals_percentage)

            leaked_games += int(leaked_deals != prev_leaked_deals)
            

            # CHECK GAME COMPLETION METRICS

            last_deal = evaluation.format_deal(evaluation.extract_deal(answers['rounds'][-1]['public_answer'], ISSUES_NUM)[0], ISSUES_NUM)
            
            # 1. Check if the last deal is feasible
            if evaluation.is_feasible(agents, last_deal):
                feasible_in_last_step += 1

            # 2. Check if the last deal is acceptable by all agents
            all_accept = all(evaluation.calculator(agents[agent]["scores"], last_deal, ISSUES_NUM, verbose=False) >= agents[agent]["scores"]["min"] for agent in agents)
            if all_accept:
                accepted_by_all_in_last_step += 1

            # 3. Check if any deal during the game was in the feasibility set
            if feasible_found:
                contained_feasible_deal += 1

        # Compute percentages
        num_games = successful_games
        perc_feasible_last = (feasible_in_last_step / num_games) * 100
        perc_accepted_all_last = (accepted_by_all_in_last_step / num_games) * 100
        perc_feasible_any = (contained_feasible_deal / num_games) * 100
        perc_wrong_deals = np.mean(wrong_deals_percentages)
        perc_leaked_deals = (leaked_deals / total_rounds) * 100
        perc_failed_games = 100 - ((successful_games / len(answers_files)) * 100)

        results[(variation, model)] = {
            "5/6-way (%)": f"{round(perc_feasible_last, 2)}",
            "6-way (%)": f"{round(perc_accepted_all_last, 2)}",
            "Any (%)": f"{round(perc_feasible_any, 2)}",
            "Wrong (%)": f"{round(perc_wrong_deals, 2)}",
            "Leakage (%)": f"{round(perc_leaked_deals, 2)}",
            "Successful games": int(successful_games),
            "Leaked games": int(leaked_games),
            "Failed games (%):": f"{round(perc_failed_games, 2)}"
        }

# Convert results dictionary to a DataFrame
results_df = pd.DataFrame.from_dict(results, orient='index')
display(results_df)


5/6-way (%) 6-way (%) Any (%)  \
original_code Llama-2-13b-chat-hf               30.0       0.0    75.0   
              Meta-Llama-3-8B-Instruct          25.0       0.0    70.0   
              Ministral-8B-Instruct-2410        25.0       0.0    50.0   
              Mixtral-8x7B-Instruct-v0.1        30.0       5.0    55.0   
              Phi-3.5-mini-instruct             10.0      10.0    50.0   
              Qwen2.5-7B-Instruct               65.0      25.0   100.0   
our_code      Llama-2-13b-chat-hf               8.33       0.0    25.0   
              Meta-Llama-3-8B-Instruct          35.0       0.0    55.0   
              Ministral-8B-Instruct-2410       22.22       0.0    50.0   
              Mixtral-8x7B-Instruct-v0.1       26.67       0.0   66.67   
              Phi-3.5-mini-instruct             35.0      10.0    95.0   
              Qwen2.5-7B-Instruct              44.44      5.56   88.89   

                                         Wrong (%) Leakage (%)  \
original_code Llama-2-13b-chat-hf            17.92        9.23   
              Meta-Llama-3-8B-Instruct        8.14       69.42   
              Ministral-8B-Instruct-2410     12.88        7.12   
              Mixtral-8x7B-Instruct-v0.1     23.64       24.81   
              Phi-3.5-mini-instruct          12.87       40.77   
              Qwen2.5-7B-Instruct            11.15       44.62   
our_code      Llama-2-13b-chat-hf            25.51        0.32   
              Meta-Llama-3-8B-Instruct         9.8         0.0   
              Ministral-8B-Instruct-2410     11.11         0.0   
              Mixtral-8x7B-Instruct-v0.1     20.14         0.0   
              Phi-3.5-mini-instruct          10.31        0.38   
              Qwen2.5-7B-Instruct             8.76         0.0   

                                          Successful games  Leaked games  \
original_code Llama-2-13b-chat-hf                       20            11   
              Meta-Llama-3-8B-Instruct                  20            20   
              Ministral-8B-Instruct-2410                20            20   
              Mixtral-8x7B-Instruct-v0.1                20            20   
              Phi-3.5-mini-instruct                     20            20   
              Qwen2.5-7B-Instruct                       20            20   
our_code      Llama-2-13b-chat-hf                       12             1   
              Meta-Llama-3-8B-Instruct                  20             0   
              Ministral-8B-Instruct-2410                18             0   
              Mixtral-8x7B-Instruct-v0.1                15             0   
              Phi-3.5-mini-instruct                     20             2   
              Qwen2.5-7B-Instruct                       18             0   

                                         Failed games (%):  
original_code Llama-2-13b-chat-hf                      0.0  
              Meta-Llama-3-8B-Instruct                 0.0  
              Ministral-8B-Instruct-2410               0.0  
              Mixtral-8x7B-Instruct-v0.1               0.0  
              Phi-3.5-mini-instruct                    0.0  
              Qwen2.5-7B-Instruct                      0.0  
our_code      Llama-2-13b-chat-hf                     40.0  
              Meta-Llama-3-8B-Instruct                 0.0  
              Ministral-8B-Instruct-2410              10.0  
              Mixtral-8x7B-Instruct-v0.1              25.0  
              Phi-3.5-mini-instruct                    0.0  
              Qwen2.5-7B-Instruct                     10.0